# Session 3 · Artefacts: detect and reject

Yesterday you cleaned SC4002 **by eye**. Today you do it **with code**, and compare the two.

You will
1. detect bad epochs with **fixed amplitude thresholds**, and pick a threshold by looking at the epochs it rejects and keeps
2. use **stage-wise thresholds** (robust z-scores)
3. use **YASA** `art_detect`
4. use the **detector** based on Mariani et al., 2018: band power relative to the neighbouring epochs
5. check the scorers' **"Movement time"** epochs, and compare **manual vs. automatic** cleaning, epoch by epoch
6. see what cleaning does to the whole-night **spectrogram**, and save the result for the next sessions.

**How to work**
- As before: `Shift+Enter` runs a cell, run from top to bottom, kernel `.venv`. Replace the `...` in `# TODO` cells.
- No interactive windows today: all plots are inline.
- AI help (chat, inline completion) is allowed. Try first yourself, and make sure you understand every line you keep.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import yasa
from scipy.signal import welch

sys.path.insert(0, str(Path("..").resolve()))  # so Python finds sleep_utils.py in the repo folder
from sleep_utils import clock_to_seconds, epoch_onsets, load_subject, make_epochs

mne.set_log_level("WARNING")

RESULTS_DIR = Path("..") / "results"
NIGHT = "SC4002"
NIGHT_DIR = RESULTS_DIR / NIGHT
NIGHT_DIR.mkdir(parents=True, exist_ok=True)

## 1. The night, in one line

From today on, `load_subject(subject, night)` (in [sleep_utils.py](../sleep_utils.py)) does all of Session 2's loading steps: find the files, fix the channel types, add and rename the hypnogram, crop from lights-off to the last sleep epoch. Open the file and have a look: it's the code you wrote yesterday.

`make_epochs(raw)` chops the night into 30-s epochs and returns the stage of each one.

In [ ]:
raw = load_subject(0, 2)  # SC4002, not filtered
raw_filt = raw.copy().filter(l_freq=0.1, h_freq=40, picks=["eeg", "eog"])  # same filter as yesterday
sfreq = raw.info["sfreq"]

epochs, stages = make_epochs(raw_filt)
onsets = epoch_onsets(epochs)  # clock time of the start of each epoch
print(epochs)
pd.Series(stages).value_counts()

### Your manual marks from yesterday

We put everything in **one table, one row per epoch**. Each detection method will add a True/False column to it.

If you have no `bad_epochs_manual.csv` (you missed Friday, or marked nothing), the next cell uses 7 example epochs instead.

The table is called `art`: one row per 30-s epoch, with its onset and its expert stage. Its first True/False column, `manual`, says whether you marked the epoch as bad yesterday.

In [ ]:
manual_file = NIGHT_DIR / "bad_epochs_manual.csv"
EXAMPLE_BAD = ["22:38:00", "01:56:30", "03:11:00", "03:50:00", "04:19:00", "04:19:30", "05:41:30"]

art = pd.DataFrame({"onset": onsets.strftime("%H:%M:%S"), "stage": stages})

manual_bad = []
if manual_file.exists():
    manual = pd.read_csv(manual_file)
    manual_bad = pd.to_datetime(manual.loc[manual["bad"], "onset"]).dt.strftime("%H:%M:%S")
    print(f"Your table: {len(manual)} epochs, {manual['bad'].sum()} marked bad")
if len(manual_bad) == 0:
    manual_bad = EXAMPLE_BAD
    print("No manual marks found: using the 7 example epochs")

art["manual"] = art["onset"].isin(manual_bad)
art.head()

### Exercise 1.1: a helper to find an epoch

We'll often want "the epoch that starts at 03:11:00". Write a function that returns its index (row number).

In [ ]:
def epoch_at(clock):
    """Index of the epoch that starts at a clock time like "03:11:00"."""
    # TODO: the position of the row where art["onset"] == clock
    # Hint: np.flatnonzero(boolean_array) returns the positions of the True values
    return ...


print(epoch_at("03:11:00"), stages[epoch_at("03:11:00")])  # should be an N1 epoch

### A helper to look at epochs

`plot_epochs` (given) plots a list of epochs with all four channels (EEG, EOG and the EMG envelope, scaled and shifted). We'll use it to check what each method rejects, e.g. `plot_epochs(["22:38:00", "01:56:30"])`.

In [ ]:
CH_PLOT = {"Fpz-Cz": 1, "Pz-Oz": 1, "horizontal": 0.5, "submental": 20}  # channel: display scale


def plot_epochs(clocks, ep=epochs):
    """One panel per epoch. Signals in µV, scaled and shifted so they don't overlap."""
    clocks = list(clocks)
    fig, axes = plt.subplots(len(clocks), 1, figsize=(12, 2.4 * len(clocks)), sharex=True, squeeze=False)
    t = ep.times
    for ax, clock in zip(axes[:, 0], clocks):
        i = epoch_at(clock)
        for k, (ch, scale) in enumerate(CH_PLOT.items()):
            x = ep.get_data(picks=ch)[i, 0] * 1e6
            ax.plot(t, scale * (x - np.median(x)) - 200 * k, lw=0.6, label=ch if clock == clocks[0] else None)
        ax.set(yticks=[], title=f"{clock} · {stages[i]}")
    axes[0, 0].legend(loc="upper right", fontsize=8, ncol=4)
    axes[-1, 0].set_xlabel("Time in epoch (s)")
    plt.tight_layout()
    plt.show()

## 2. Fixed amplitude thresholds

The simplest automatic rule: reject an epoch if its **peak-to-peak** amplitude (max − min) is above a threshold, on any EEG channel.

### Exercise 2.1: peak-to-peak per epoch

In [ ]:
eeg = epochs.get_data(picks=["Fpz-Cz", "Pz-Oz"]) * 1e6  # (n_epochs, 2, n_samples), µV
# TODO: peak-to-peak of each epoch and channel: (n_epochs, 2). Hint: .max(axis=-1) and .min(axis=-1)
ptp = ...
# TODO: per epoch, the larger of the two channels
art["ptp_max"] = ...

order = ["WAKE", "N1", "N2", "N3", "REM"]
fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot([art.loc[art["stage"] == s, "ptp_max"] for s in order], tick_labels=order)
ax.set(ylabel="Peak-to-peak (µV)", title="Largest peak-to-peak per epoch (Fpz-Cz or Pz-Oz)")
plt.show()

### Exercise 2.2: try some thresholds

For each threshold, count the epochs that would be rejected, **per stage**.

In [ ]:
rows = {}
for thr in [200, 250, 300, 500]:
    # TODO: the number of epochs per stage with ptp_max above thr. Hint: art.loc[condition, "stage"].value_counts()
    rows[thr] = ...
pd.DataFrame(rows).T.reindex(columns=order).fillna(0).astype(int).rename_axis("threshold (µV)")

**Question:** Which stage is hit hardest by a low threshold? Is that because it has more artefacts?

### Exercise 2.3: choose your threshold

Counts alone don't tell you whether a threshold is good: you have to **look** at what it rejects. The cell below shows the epochs just **above** the threshold (the "least bad" ones it rejects) and just **below** it (the "worst" ones it keeps).

👉 Change `THRESHOLD`, rerun the cell, and repeat until you find the value you would defend. Try at least 200, 300 and 400. Which threshold would you choose?

In [ ]:
THRESHOLD = 300  # µV peak-to-peak: change me and rerun
N_EXAMPLES = 3

# TODO: True for the epochs with ptp_max above THRESHOLD
art["fixed"] = ...
print(f"{THRESHOLD} µV: {art['fixed'].sum()} epochs rejected ({100 * art['fixed'].mean():.1f} %)")
print(art.loc[art["fixed"], "stage"].value_counts().reindex(order, fill_value=0).to_dict())

# TODO: the N_EXAMPLES rejected epochs with the smallest ptp_max, and the N_EXAMPLES kept epochs with the largest
# Hint: art.loc[art["fixed"]].nsmallest(N_EXAMPLES, "ptp_max"), and .nlargest(...) for the kept ones (~art["fixed"])
just_above = ...
just_below = ...

if len(just_above):
    print("Just ABOVE the threshold (rejected):", just_above["ptp_max"].round().tolist(), "µV")
    plot_epochs(just_above["onset"])
else:
    print("Nothing above the threshold: no epoch is rejected")
if len(just_below):
    print("Just BELOW the threshold (kept):", just_below["ptp_max"].round().tolist(), "µV")
    plot_epochs(just_below["onset"])
else:
    print("Nothing below the threshold: every epoch is rejected")

**Questions**
1. Which threshold did you choose, and why? How many epochs does it reject, and in which stages?
2. Is there a threshold that rejects the clear artefacts but keeps all clean N3?
3. Many papers reject epochs above **±500 µV**. What would that do on this dataset?

The rest of the notebook uses the `fixed` column with the threshold you ran last (the solution uses 300 µV).

## 3. Stage-wise thresholds: robust z-scores

Idea: an epoch is bad if it is **unusual for its own stage**. For each stage, we measure how far each epoch is from the typical epoch of that stage, in units of spread: a **z-score**.

Two details:
- We use the **median** and the **MAD** (median absolute deviation) instead of mean and SD. Artefacts are extreme values: they would pull the mean and SD towards themselves and hide. The median and MAD barely move.
- Amplitudes and powers are skewed (a few huge values), so we take the **log** first.

Two features:
- **amplitude**: log of the SD of each epoch (movement, electrode)
- **muscle**: log of the mean power at 20–40 Hz (EMG)

In [ ]:
def robust_z(x):
    """(x - median) / (1.4826 * MAD). The 1.4826 makes it comparable to a normal z-score."""
    x = np.asarray(x, dtype=float)
    med = np.median(x)
    mad = 1.4826 * np.median(np.abs(x - med))
    return (x - med) / mad

### Exercise 3.1: features per epoch

In [ ]:
# TODO: log10 of the SD of each epoch and channel: (n_epochs, 2)
# Hint: eeg.std(axis=-1) is the SD of each epoch and channel; np.log10(...) takes the log of every value
log_sd = ...

# Given: the power at each frequency, then the mean between 20 and 40 Hz
f, p = welch(eeg, sfreq, nperseg=int(4 * sfreq))  # (n_epochs, 2, n_freqs)
log_muscle = np.log10(p[..., (f >= 20) & (f <= 40)].mean(axis=-1))  # (n_epochs, 2)

for k, ch in enumerate(["fpz", "pz"]):
    art[f"log_sd_{ch}"] = log_sd[:, k]
    art[f"log_muscle_{ch}"] = log_muscle[:, k]
art.head()

### Exercise 3.2: z-score within each stage

`groupby("stage")[columns].transform(robust_z)` applies `robust_z` to each stage separately and returns a table of the same shape. `ART` and `UNS` epochs are left out: they are not a stage (Section 6).

In [ ]:
features = ["log_sd_fpz", "log_sd_pz", "log_muscle_fpz", "log_muscle_pz"]
in_stage = art["stage"].isin(order)

# TODO: the robust z-score of each feature, within each stage (only the rows in_stage)
z = ...

Z_THR = 3
# TODO: True if |z| > Z_THR for the amplitude of either channel
art["z_amplitude"] = (...).any(axis=1).reindex(art.index, fill_value=False)
# TODO: True if z > Z_THR (only too high!) for the muscle power of either channel
art["z_muscle"] = (...).any(axis=1).reindex(art.index, fill_value=False)
art[["z_amplitude", "z_muscle"]].sum()

**Questions**
1. For amplitude we reject |z| > 3 (too large *or* too small), for muscle only z > 3. Why? What could a far-too-small amplitude mean?
2. Look at the epochs flagged **only** by `z_muscle` (next cell): the strongest ones, and the ones just above the threshold. Would you reject them?

In [ ]:
z_mus = z[["log_muscle_fpz", "log_muscle_pz"]].max(axis=1)  # the larger muscle z of the two channels
only_muscle = z_mus[(art["z_muscle"] & ~art["z_amplitude"]).loc[z_mus.index]].sort_values()
print(len(only_muscle), "epochs flagged only by z_muscle")

print("The 3 strongest:")
plot_epochs(list(art.loc[only_muscle.index[-3:], "onset"]))
print("The 3 closest to the threshold:")
plot_epochs(list(art.loc[only_muscle.index[:3], "onset"]))

## 4. YASA `art_detect`

YASA has a ready-made detector with the same idea: per-stage z-scores, computed per window. Two methods:
- `"covar"`: compares the **covariance between channels** to the typical one (Riemannian geometry). Needs **≥ 4 EEG channels**.
- `"std"`: z-score of the SD of each channel, as in Section 3 (but not robust and not log).

With only 2 EEG channels, YASA can only use `"std"`.

YASA's windows must match our epochs exactly, so we glue the epochs back together and give YASA the hypnogram as a `yasa.Hypnogram`.

### Exercise 4.1

The documentation of `yasa.art_detect`, with all its arguments: https://yasa-sleep.org/generated/yasa.art_detect.html

In [ ]:
data = np.concatenate(eeg, axis=-1)  # (2, n_epochs * 3000): the epochs, glued back together
hypno = yasa.Hypnogram(stages, freq="30s")

# TODO: run yasa.art_detect on `data` with sf=sfreq, window=30 (s), the hypnogram, and all five stages in `include`
# Use method="std" and threshold=3. It returns two arrays: True/False per window, and the z-scores
# Hint: is_art, zscores = yasa.art_detect(data, sf=..., window=..., hypno=..., include=[...], method=..., threshold=...)
is_art, zscores = ...
art["yasa"] = is_art
print(art["yasa"].sum(), "epochs flagged")
art.loc[art["yasa"], "stage"].value_counts()

**Question:** By default `include` has only the sleep stages (`N1`–`REM`). What happens to WAKE epochs then? Why could that be a sensible default?

## 5. The detector based on Mariani et al., 2018: power relative to the neighbouring epochs

All detectors so far compare an epoch with **all** epochs of its stage. The detector we use from now on in this course compares it with its **neighbours in time**:

1. For each epoch, compute the mean power in a **slow** band (0.5–4.5 Hz: movement, sweat, electrode drifts) and a **fast** band (20–40 Hz: muscle), averaged over the EEG channels.
2. For each epoch, divide its power by the **mean power of the surrounding epochs** (here ±7 epochs = ±3.5 min, the epoch itself included).
3. Reject the epoch if the slow ratio is above 2.5 or the fast ratio is above 2.

Why neighbours? The EEG changes slowly over a night (sleep depth, electrode contact drying out, sweating), and an artefact is a **sudden** change on top of that. This needs **no hypnogram**, so it also works on unscored data, or on rodent recordings before scoring.

The method follows the automated artefact rejection of the National Sleep Research Resource (NSRR) pipeline:
> Mariani S, Tarokh L, Djonlagic I, Cade BE, Morrical MG, Yaffe K, Stone KL, Loparo KA, Purcell SM, Redline S, Aeschbach D (2018). Evaluation of an automated pipeline for large-scale EEG spectral analysis: the National Sleep Research Resource. *Sleep Medicine* 47:126–136. [doi:10.1016/j.sleep.2017.11.1128](https://doi.org/10.1016/j.sleep.2017.11.1128)

⚠️ **Rejected epochs are removed from the EEG analyses only.** The sleep stages stay as the experts scored them.

In [ ]:
SLOW_BAND = (0.5, 4.5)   # Hz: movement, sweat, electrode drifts
FAST_BAND = (20, 40)     # Hz: muscle
WINDOW = 7               # neighbouring epochs on each side (7 x 30 s = 3.5 min)
THR_SLOW = 2.5           # reject if the slow power is > 2.5x the local mean
THR_FAST = 2             # reject if the fast power is > 2x the local mean

The detector uses the **power spectrum** of each epoch: the power at each frequency, averaged over the 30 s. We compute it with Welch's method, as for yesterday's spectrogram.

In [ ]:
def epoch_psd(ep, ch="Fpz-Cz"):
    """Welch PSD of every epoch: freqs, power (n_epochs, n_freqs) in µV²/Hz."""
    x = ep.get_data(picks=ch)[:, 0, :] * 1e6
    return welch(x, sfreq, nperseg=int(4 * sfreq))


freqs, psd = epoch_psd(epochs)

### Exercise 5.1: band power of one spectrum

The function gets the frequencies and the power spectrum of **one** epoch, and returns the mean power in the two bands. The indices of the band edges are given.

In [ ]:
def band_power_artifact_rejection(freqs, psd, slow_band=SLOW_BAND, fast_band=FAST_BAND):
    """Mean power in the slow and in the fast artefact band of one spectrum."""
    idx_freq_lower_slow = np.where(freqs >= slow_band[0])[0][0]
    idx_freq_higher_slow = np.where(freqs <= slow_band[1])[0][-1]
    idx_freq_lower_fast = np.where(freqs >= fast_band[0])[0][0]
    idx_freq_higher_fast = np.where(freqs <= fast_band[1])[0][-1]

    # TODO: the mean of psd between the two slow-band indices, and between the two fast-band indices
    power_slow = ...
    power_fast = ...

    return {"power_slow": power_slow, "power_fast": power_fast}


band_power_artifact_rejection(freqs, psd[epoch_at("01:56:30")])  # one of the example artefacts (muscle + movement)

### Exercise 5.2: compare each epoch with its neighbours

At the start and the end of the night there are fewer than 7 neighbours on one side: the window must be cut at the first and the last epoch.

In [ ]:
def artifact_rejection(epo, picks=("Fpz-Cz", "Pz-Oz"), window=WINDOW, thr_slow=THR_SLOW, thr_fast=THR_FAST,
                       return_ratios=False):
    """Indices of the epochs whose slow or fast power is > threshold x the mean of the surrounding epochs.

    Based on the NSRR pipeline: Mariani et al. (2018), Sleep Medicine 47:126-136,
    https://doi.org/10.1016/j.sleep.2017.11.1128

    Use the result to drop epochs from EEG-based features (spectra, band power, spindles, slow waves).
    Do NOT edit the hypnogram with it: sleep architecture statistics use the unedited expert hypnogram.
    """
    sf = epo.info["sfreq"]
    data = epo.get_data(picks=list(picks))  # EEG only: (n_epochs, n_channels, n_samples), in V
    n_epos = len(data)

    # Power in the slow and fast bands of every epoch
    powers_slow = np.zeros(n_epos)
    powers_fast = np.zeros(n_epos)
    for i, epoch_data in enumerate(data):
        freqs, psds = welch(epoch_data, sf, nperseg=int(4 * sf))  # (n_channels, n_freqs)
        psd = np.mean(psds, axis=0)  # mean over the channels
        yy = band_power_artifact_rejection(freqs, psd)
        powers_slow[i] = yy["power_slow"]
        powers_fast[i] = yy["power_fast"]

    # Ratio of each epoch's power to the mean of its neighbours
    slow_ratios = np.zeros(n_epos)
    fast_ratios = np.zeros(n_epos)
    for i in range(n_epos):
        # TODO: the first epoch of the window (i - window), but not below 0
        idx_low = ...
        # TODO: one after the last epoch of the window (i + window + 1), but not above n_epos
        idx_high = ...

        # TODO: the power of epoch i divided by the mean power of the window, for both bands
        slow_ratios[i] = ...
        fast_ratios[i] = ...

    artifacts = np.unique(np.concatenate((np.where(fast_ratios > thr_fast)[0],
                                          np.where(slow_ratios > thr_slow)[0])))
    if return_ratios:
        return artifacts, slow_ratios, fast_ratios
    return artifacts


bad_idx, slow_ratios, fast_ratios = artifact_rejection(epochs, return_ratios=True)
# TODO: a True/False column: True for the epochs in bad_idx. Hint: np.isin(np.arange(len(art)), bad_idx)
art["custom"] = ...
print(art["custom"].sum(), "epochs flagged")
art.loc[art["custom"], "stage"].value_counts()

### Exercise 5.3: the ratios across the night

In [ ]:
times_h = np.arange(len(art)) * 30 / 3600  # epoch start, h after lights-off

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
for ax, ratios, thr, name in [(axes[0], slow_ratios, THR_SLOW, "slow (0.5-4.5 Hz)"),
                              (axes[1], fast_ratios, THR_FAST, "fast (20-40 Hz)")]:
    # TODO: plot the ratios over time, a dashed horizontal line at thr (ax.axhline), and mark the epochs above thr
    ...
    ax.set(ylabel="Power / local mean", title=name)
axes[1].set_xlabel("Time after lights-off (h)")
plt.show()

**Questions**
1. Plot the flagged epochs with `plot_epochs(list(art.loc[art["custom"], "onset"]))`. Are they artefacts?
2. Which of the two bands does the work in this night? Why do the slow ratios stay so low?
3. When could comparing with the neighbours go wrong? Think of (a) an artefact that lasts 10 minutes, (b) the first epochs of N3 after a long stretch of N2, (c) a bad epoch next to another bad epoch.

## 6. "Movement time": the scorers' own flag

R&K scorers labelled epochs dominated by movement as **Movement time** (`ART` here). These have no sleep stage, so the per-stage methods (Sections 3 and 4) never look at them.

### Exercise 6.1

In [ ]:
# TODO: True for the epochs the scorers labelled as movement (stage "ART")
art["movement"] = ...

print(art["movement"].sum(), "movement epoch(s)")
art.loc[art["movement"], ["onset", "stage", "manual", "fixed", "z_amplitude", "z_muscle", "yasa", "custom"]]

**Question:** Did any automatic method catch the movement epoch? What does that tell you?

## 7. Manual vs. automatic

Now we have seven True/False columns. First: how much does each method reject, per stage?

### Exercise 7.1: % rejected per stage and method

In [ ]:
METHODS = ["manual", "fixed", "z_amplitude", "z_muscle", "yasa", "custom", "movement"]

# TODO: per stage, the % of epochs flagged by each method
# Hint: the mean of a True/False column is the fraction of True
pct = ...
pct.loc["all"] = ...
pct.round(1)

### Exercise 7.2: what does each one miss?

Pick one method (`AUTO`) and compare it with your marks: which epochs did **you** mark but `AUTO` did not, and the other way round? The cell plots up to 3 of each.

In [ ]:
AUTO = "custom"  # try the others too

# TODO: the onsets of the epochs marked by you but not by AUTO, and the other way round
# Hint: art.loc[art["manual"] & ~art[AUTO], "onset"]; ~ means "not"
only_manual = ...
only_auto = ...
print("Only manual:", list(only_manual))
print(f"Only {AUTO}:", list(only_auto))

if len(only_manual):
    plot_epochs(list(only_manual[:3]))
if len(only_auto):
    plot_epochs(list(only_auto[:3]))

**Questions** (for the debrief)
1. Which method agrees best with you? Is your manual marking the "truth"?
2. What do the automatic methods catch that you missed, and the other way round?
3. Which would you trust on 153 nights?

## 8. The final rule, and its effect on the spectrogram

There is no single right answer, but we need one rule for the rest of the course (the next sessions use this file). Ours: the **detector based on Mariani et al., 2018** (Section 5) plus the scorers' **movement epochs**.

### Exercise 8.1

In [ ]:
# TODO: the course rule: bad if flagged by the custom detector OR a movement epoch. Hint: column1 | column2
art["bad"] = ...

print(f"Rejected: {art['bad'].sum()} of {len(art)} epochs ({100 * art['bad'].mean():.1f} %)")
art.groupby("stage")["bad"].mean().mul(100).round(1)

### Exercise 8.2: the whole-night spectrogram, before and after

The same figure as yesterday: one column per epoch, colour = power in dB, rejected epochs in white. `psd` (Section 5) already has the power of every epoch at every frequency on Fpz-Cz.

In [ ]:
TFR_CMAP = plt.get_cmap("Spectral_r").copy()  # blue = low power, red = high power
TFR_CMAP.set_bad("white")  # rejected epochs (NaN)


def plot_hypnogram(stages, times_h, ax):
    """The expert hypnogram as a step line, one value per epoch."""
    order = {"WAKE": 0, "REM": 1, "N1": 2, "N2": 3, "N3": 4}
    y = np.array([order.get(s, np.nan) for s in stages])
    ax.step(times_h, y, where="post", color="#0b0b0b", lw=1)
    ax.set_yticks(list(order.values()), list(order.keys()))
    ax.invert_yaxis()
    ax.set(xlabel="Time after lights-off (h)", xlim=(0, times_h[-1] + 30 / 3600))


times_h = (epochs.events[:, 0] / sfreq - raw_filt.first_time) / 3600  # epoch start, h after lights-off
power_before = 10 * np.log10(psd)  # (n_epochs, n_freqs), dB
# TODO: a copy of power_before with the bad epochs (rows) set to np.nan. Hint: art["bad"].to_numpy()
power_after = ...

vmin, vmax = np.nanpercentile(power_before[:, freqs <= 30], [2, 98])  # the same colour range for both panels

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True, height_ratios=[3, 3, 1.2])
for ax, power, title in [(axes[0], power_before, "Before cleaning"), (axes[1], power_after, "After cleaning")]:
    mesh = ax.pcolormesh(times_h, freqs, power.T, cmap=TFR_CMAP, vmin=vmin, vmax=vmax, shading="nearest")
    ax.set(ylim=(0.5, 30), ylabel="Frequency (Hz)", title=f"{title}: Fpz-Cz")
fig.colorbar(mesh, ax=axes, shrink=0.7, label="Power (dB µV²/Hz)")
plot_hypnogram(art["stage"], times_h, axes[2])
plt.show()

**Questions**
1. Where in the night are the rejected epochs? Compare with the hypnogram.
2. What did the rejected epochs look like in the "before" panel? Compare with your manual cleaning from yesterday.
3. What does this mean for the band powers you will compute on Monday?

### Exercise 8.3: save it

In [ ]:
art_file = NIGHT_DIR / "bad_epochs_auto.csv"
# TODO: save the columns onset, stage, the METHODS and bad to art_file (without the index)
...
pd.read_csv(art_file).head()

## Optional exercises

1. **One function for any night.** Write `detect_artefacts(raw)` that returns the table of Section 8 (onset, stage, `movement`, `z_amplitude`, `z_muscle`, `yasa`, `custom`, `bad`) for any night. Run it on night 1 (`load_subject(0, 1)`) and on an older subject (e.g. `load_subject(80, 1)` if you have it). Is the % rejected per stage similar? *(You'll need this function on Tuesday.)*
2. **Tune the threshold, and the trap.** Run `artifact_rejection` with thresholds from 2 to 6 (the same for both bands). For each one, count how many of your manual marks it finds and how many epochs it flags that you didn't mark. Which threshold "wins"? Why would choosing it this way on one night, and then reporting how well it does on the same night, be too optimistic?

In [ ]:
# Optional exercise 1

In [ ]:
# Optional exercise 2